## ML Project : Diabetes Prediction Using Random Forest Classifier

In [ ]:
import sys
! {sys.executable} -m pip install pandas numpy scikit-learn seaborn matplotlib

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, roc_curve, classification_report
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt
import seaborn as sns

### Step 1: Data Gathering

In [ ]:
df = pd.read_csv('Diabetes.csv')
df.head(10)

### Step 2:EDA - Read and explore the data

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df['Outcome'].value_counts()

In [ ]:
def train_test_split_and_features(df):
    y = df["Outcome"]
    x = df.drop(['Outcome','Id'],axis=1)
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state = 0)
    print(x.head(5))
    print(x.columns)
    features = list(x.columns)
    return x_train, x_test, y_train, y_test,features

In [ ]:
x_train, x_test, y_train, y_test,features = train_test_split_and_features(df)

Random forest classifiers have proven effective in predicting diabetes by analyzing various factors such as patient demographics, medical history, and biomarkers. By leveraging the ensemble learning technique and combining multiple decision trees, random forest models can provide accurate predictions of diabetic risk, aiding in early detection and personalized treatment strategies.

### Step 4: Fit and evaluate model

In [ ]:
def fit_and_evaluate_model(x_train, x_test, y_train, y_test,max_depth=5,min_samples_split=0.01,max_features=0.8,max_samples=0.8):
    random_forest =  RandomForestClassifier(random_state=0,\
                                            max_depth=max_depth,\
                                            min_samples_split=min_samples_split,\
                                            max_features=max_features,
                                            max_samples=max_samples)

    model = random_forest.fit(x_train, y_train)
    random_forest_predict = random_forest.predict(x_test)
    random_forest_conf_matrix = confusion_matrix(y_test, random_forest_predict)
    random_forest_acc_score = accuracy_score(y_test, random_forest_predict)
    print("confussion matrix")
    print(random_forest_conf_matrix)
    print("\n")
    print("Accuracy of Random Forest:",random_forest_acc_score*100,'\n')
    print(classification_report(y_test,random_forest_predict))
    return model

In [ ]:
model = fit_and_evaluate_model(x_train, x_test, y_train, y_test)

### Step 5: Try different parameters to identify best ones

In [ ]:
param_grid = [
  {'max_depth': [3,5,7,10], 'min_samples_split': [0.01,0.03, 0.07, 0.1], 
'max_features': [0.7,0.8,0.9,1.0],
'max_samples': [0.7,0.8,0.9,1.0]}]

In [ ]:
model = RandomForestClassifier()
search = GridSearchCV(estimator = model, param_grid = param_grid, cv=5, verbose=5)
search.fit(x_train, y_train)

In [ ]:
results = pd.DataFrame(search.cv_results_)
results.sort_values('mean_test_score',inplace=True,ascending= False)
results.head(10)

In [ ]:
results_save = pd.DataFrame(search.cv_results_)
results_save.to_csv("results_save.csv", index =False)

In [ ]:
search.best_params_

### Step 6: Evaluate the model with best parameters

In [ ]:
model = fit_and_evaluate_model(x_train, x_test, y_train, y_test, max_depth=10,min_samples_split=0.01,\
                               max_features=1.0,max_samples= 0.9)

In [ ]:
importances = pd.DataFrame(model.feature_importances_)
importances['features'] = features
importances.columns = ['importance','feature']
importances.sort_values(by = 'importance', ascending= True,inplace=True)

In [ ]:
plt.figure(figsize=(5, 3))
plt.barh(importances.feature, importances.importance)